In [ ]:
from ultralytics import YOLO
from pathlib import Path
from collections import Counter

In [ ]:
ROOT_FOLDER = "slide_folder_path"
MODEL_PATH = "yolo_model_weights.pt"
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

CONF_THRESHOLD = 0.25
COUNT_ALL_BOXES = True

ATYPICAL_IDX = 1
TYPICAL_IDX = 2

In [ ]:
model = YOLO(MODEL_PATH)
class_names = model.names

In [ ]:
root = Path(ROOT_FOLDER)

if not root.exists():
    raise FileNotFoundError(f"Root folder not found: {ROOT_FOLDER}")

slide_folders = [f for f in root.iterdir() if f.is_dir()]

if not slide_folders:
    print("No slide subfolders found.")
    raise SystemExit

In [ ]:
if isinstance(class_names, dict):
    ordered_class_names = [class_names[i] for i in sorted(class_names.keys())]
else:
    ordered_class_names = list(class_names)

ordered_class_names

In [ ]:
for slide_folder in sorted(slide_folders):
    counts = Counter()

    image_paths = [
        p for p in slide_folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]

    if not image_paths:
        print(f"\nSlide: {slide_folder.name}")
        print("  No images found.")
        continue

    for img_path in image_paths:
        results = model.predict(
            source=str(img_path),
            conf=CONF_THRESHOLD,
            verbose=False
        )

        result = results[0]

        if result.boxes is None or len(result.boxes) == 0:
            continue

        classes = result.boxes.cls.cpu().numpy().astype(int)
        confidences = result.boxes.conf.cpu().numpy()

        if COUNT_ALL_BOXES:
            for cls_id in classes:
                counts[cls_id] += 1
        else:
            best_idx = confidences.argmax()
            best_cls = classes[best_idx]
            counts[best_cls] += 1

    print(f"\nSlide: {slide_folder.name}")

    for cls_id, cls_name in enumerate(ordered_class_names):
        print(f"  {cls_name}: {counts.get(cls_id, 0)}")

    atypical_count = counts.get(ATYPICAL_IDX, 0)
    typical_count = counts.get(TYPICAL_IDX, 0)
    denom = atypical_count + typical_count

    if denom == 0:
        ratio = None
        print("  atypical_ratio: undefined (no atypical or typical cells detected)")
    else:
        ratio = atypical_count / denom
        print(f"  atypical_ratio: {ratio:.4f}")

In [ ]:
import pandas as pd

rows = []

for slide_folder in sorted(slide_folders):
    counts = Counter()

    image_paths = [
        p for p in slide_folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]

    if not image_paths:
        row = {"slide": slide_folder.name}
        for cls_id, cls_name in enumerate(ordered_class_names):
            row[cls_name] = 0
        row["atypical_ratio"] = None
        rows.append(row)
        continue

    for img_path in image_paths:
        results = model.predict(
            source=str(img_path),
            conf=CONF_THRESHOLD,
            verbose=False
        )

        result = results[0]

        if result.boxes is None or len(result.boxes) == 0:
            continue

        classes = result.boxes.cls.cpu().numpy().astype(int)
        confidences = result.boxes.conf.cpu().numpy()

        if COUNT_ALL_BOXES:
            for cls_id in classes:
                counts[cls_id] += 1
        else:
            best_idx = confidences.argmax()
            best_cls = classes[best_idx]
            counts[best_cls] += 1

    row = {"slide": slide_folder.name}
    for cls_id, cls_name in enumerate(ordered_class_names):
        row[cls_name] = counts.get(cls_id, 0)

    atypical_count = counts.get(ATYPICAL_IDX, 0)
    typical_count = counts.get(TYPICAL_IDX, 0)
    denom = atypical_count + typical_count
    row["atypical_ratio"] = None if denom == 0 else atypical_count / denom

    rows.append(row)

results_df = pd.DataFrame(rows)
results_df

In [ ]:
results_df.to_csv("slide_level_counts.csv", index=False)